# CSMAR 实战：用 Codex 从 ZIP 建立年度数据库 {#sec-csmar-case}

这是数据工作流单元的完整案例。先阅读[流程与获取](../../lectures/financial-data.qmd)、[数据管理](../../lectures/financial-data-management.qmd)和[清洗与审计](../../lectures/financial-data-cleaning.qmd)。本案例重点是把研究规则变成 Agent 可以执行的任务，再用实际输出核验。

学生可以 clone [FinEco 仓库](https://github.com/lianxhcn/FinEco)，按[运行说明](../../data/financial-data/README.md)配置环境与授权数据，在本地运行这份 Notebook。没有 CSMAR 数据也能单独运行末尾的[教学小表](#sec-csmar-mini)。

本实验从授权 ZIP 开始，逐步建立标准化表、候选年度底表、变量比较与审计结果。输出仅包含汇总统计；完整记录留在本地。当前数据中的公司身份、上市日期及历史状态还有待核对，**候选底表不等于可直接用于论文的最终样本**。

依次运行 00–11。教师提供的坚果云下载入口、校验值和解压步骤见 [CSMAR 数据获取](../../data/financial-data/DATA_ACCESS.md)。仓库不附原始数据，请在已有授权范围内使用。

## 开始执行前：建立规则文件 {#sec-csmar-rules}

本案例从一组真实授权 ZIP 出发。给 Agent 一句清洗数据还不足以开始，需要先明确它能读写哪里、哪些口径已有依据、遇到什么情况必须报告。

课程仓库现有根规则继续有效。下面模板供学生自己的独立项目使用，不能覆盖教师仓库规则。将 AGENTS.example.md 在学生项目另存为 AGENTS.md，将 DECISIONS.template.md 另存为 DECISIONS.md，再核对项目路径。项目计划说明阶段，数据协议说明含义与允许动作，提示词只安排本次任务。

| 文档 | 下载与用途 |
|---|---|
| Agent 工作规则 | [AGENTS.example.md](../../data/financial-data/agent-project/AGENTS.example.md)，规定工作边界 |
| 数据协议 | [DATA_PROTOCOL.md](../../data/financial-data/DATA_PROTOCOL.md)，定义四层规则 |
| 项目阶段 | [PROJECT_PLAN.md](../../data/financial-data/agent-project/PROJECT_PLAN.md)，列交付和验收 |
| 全套任务提示词 | [PROMPTS.md](../../data/financial-data/agent-project/PROMPTS.md)，P00–P07 |
| 人工决策 | [DECISIONS.template.md](../../data/financial-data/agent-project/DECISIONS.template.md)，待决与批准分开 |

这些提示词是依据已执行工作编写的教学模板，后面的程序是本案例实际实现，不能据此声称复制同一句提示词必然生成完全相同代码。

### 工作规则模板全文

```markdown
# CSMAR 学生项目：Agent 工作规则模板

使用方式：在学生自己的独立项目中另存为 `AGENTS.md`，先由学生填写并核对项目路径。本文件在课程仓库中是教学模板，不覆盖 FinEco 现有规则。

## 每次任务先读

阅读本文件、`PROJECT_PLAN.md`、`DATA_PROTOCOL.md`、`DECISIONS.md` 和当前任务提示词。遇到冲突先报告，不能把模板中的建议写成教师已经批准的规则。

## 工作边界

- 原始 ZIP 与解压原件只读；核对授权输入根目录，派生结果写入本次运行目录。
- 只修改指定学生项目，开始时列出允许修改的文件。不得改写教师来源仓库、系统环境或共享配置。
- 不读取凭据，不公开受限记录，不将公司级原始或派生数据上传 GitHub、网页或外部服务。
- 字段含义必须对应提供方说明；单位、币种、历史状态或身份不明确时保持待确认。
- 不自动填补、缩尾、删除行业或状态样本，不为了唯一性任意保留一条记录。
- 合并前核对粒度、空键与重复键；连接关系不符合设计时停止相应步骤。
- 代码改变后从头运行，保存版本和真实输出。不能手工编造通过状态或 Notebook 结果。
- 每次交付报告输入、规则、代码、输出、检查与待决事项，并区分执行成功与研究口径获批。

## 审核接口

没有依据的事项输出 `HUMAN REVIEW REQUIRED`，附证据、影响及需要谁作何种决定。仅将已获得教师明确确认的规则写入 `DECISIONS.md` 的批准栏，其余留在待决栏。

```

### 项目计划


目标：按已确认规则，从获授权 CSMAR 导出文件建立可审计年度候选库。研究期为 2015–2025，2014 年财务用于相邻年度分母。最终粒度目标为公司年度，当前证券年度结果必须另行核对身份。

| 阶段 | 交付 | 进入下一步的条件 |
|---|---|---|
| 输入盘点 | Source Manifest、字段说明清单 | ZIP 可读、来源明确、授权和路径确认 |
| 结构与标准化 | schema、映射、interim、转换失败汇总 | 保留原始行；字段定义有据；失败已说明 |
| 年度视图 | 期间与 A/B 样本流 | 每个过滤条件可追溯，其他原行仍在 interim |
| 多表合并 | 候选底表、merge audit、身份冲突表 | 证券年度键检查通过；公司身份问题单列 |
| 变量构造 | 原始分子分母、候选比率 | 单位与口径未确认时清楚标为条件性候选 |
| 审计返修 | 分布、缺失、会计、时间和分组审计 | 发现的问题已对应返修或人工裁定 |
| 版本保存 | manifest、代码哈希、环境、决策状态 | 区分已保存证据与最终研究数据批准 |

按顺序采用 `PROMPTS.md`。发现一个阶段的未决问题时，可继续互不依赖的检查，但不能让待定信息影响的下游结果被标为正式通过。当前案例的实际未决事项见上一级 `DATA_GAPS.md`。


### 本案例的数据协议

下面给出完整协议，课堂可按需讲授，学生不用跳到另一份讲义寻找关键边界。

### 金融数据章 Data Agent Protocol

版本：v2 本地整合稿。retrieved_date: 2026-09-16。教师依据：本次任务书及当前对话确认。执行实现：`tools/financial-data/pipeline.py`；课堂入口：`notebooks/financial-data/financial-data.ipynb`。

#### 1. Global Rules

1. 不修改、覆盖或重命名原始 ZIP；解压文件按数据集隔离，已有内容必须与原 ZIP 一致。
2. 不静默删除观测。标准化保留全部行，年度视图筛选必须记录前后行数和理由，未纳入行仍在支持层或候选层。
3. 不按变量代码猜定义。先读本批 TXT、数据库说明，再形成字段映射。
4. 单位改变须有字典依据，保留原值及转换式。财务金额单位未证实时不声明为人民币元。
5. 不自动填补缺失，不自动缩尾，不自行选择论文研究样本。
6. 合并前验证两侧粒度、键完整性及唯一性；不用未经说明的 many-to-many。
7. A/B 报表分开保存；指标命名标明口径。不能把归母净利润与全部权益的混合口径称为唯一 ROE。
8. 相邻年度滞后按明确的年度键匹配，不能跨断档顺序移位。
9. 程序运行成功只通过执行检查；经济逻辑、身份映射和时间信息另行审核。
10. 无法裁定的事项标为 HUMAN REVIEW REQUIRED，保留证据，不编造确认结果。
11. 记录级数据只写受限目录；公开输出实行文件允许清单，不展示真实公司行。
12. 冻结输入指纹、代码指纹、环境版本与执行时间；更新数据后重新运行审计。

#### 2. Project Rules

| 项目 | 规则与当前实现 |
|---|---|
| Population | 目标为 2015–2025 年间曾上市的 A 股公司；当前下载是否覆盖完整历史总体尚待核验 |
| Raw period | 2014–2025 为年度视图；原文件还含 2026 季报及 1 月 1 日记录，标准化层完整保存 |
| Unit | 目标 firm-year；当前技术键 stock_code + fiscal_year，另检验 firm_id + fiscal_year |
| A 股身份 | TRD_Year 的市场编码 1、4、16、32、64 提供已观测身份；缺少证据的代码保留在 candidate |
| Listing | 年度表的首次上市日期为候选口径，与 CG_Co 的上市日期逐项比较；冲突标识不能视为已裁定 |
| Master | 年度来源键先 outer 合并；候选视图再限制研究年、可确认 A 股身份及已知上市期间；完整并集保留 |
| Financial industry | 根据年度行业 J 前缀标识，不删除；行业标准 2023 年发生变化 |
| ST/PT | 保留源状态及标识，不删除；源定义为已出年报最新状态，完整历史状态尚未核验 |
| Missing / winsor | 不填补，不缩尾；零分母比率缺失，负分母原样保留并诊断 |
| Statement type | A 合并报表与 B 母公司报表分别并入；主案例比较合并报表口径，母公司列保留 |
| Announcement | 使用 Actudt；Firforecdt 只作预约披露日期，不替代缺失 Actudt |
| Export | 环境变量 FINECO_CSMAR_ROOT；记录级输出进入根目录下 financial-data-v2/runs/ |
| Public status | Notebook 汇总输出可审核；最终公司年度 master 尚未验收，不宣称可直接做论文 |

这里的上市期间判定是可复现候选规则。上市日期冲突、退市覆盖不足及证券代码迁移均会影响最终总体，不能被候选视图的程序名掩盖。

#### 3. Task Rules：可复制给 Agent 的五个任务

##### 3.1 检查资产负债表的键

本任务解决“一行究竟代表什么”。

- Input：原始 FS_Combas CSV、同包 TXT。
- Goal：验证 Stkcd + Accper + Typrep，比较 Stkcd + 年度的重复情况。
- Allowed actions：读取、统计、保存受限重复组，形成汇总报告。
- Forbidden actions：删除重复、取第一条、选择 A/B、改变日期。
- Required checks：键缺失、重复、日期后缀、A/B 频数、原始行数。
- Expected outputs：schema.json、候选键结论、报告期分布。
- Human review conditions：字段无定义、完整键不唯一或日期无法解释。

##### 3.2 标准化资产负债表

本任务解决字段类型和名称的一致性。

- Input：通过来源检查的 CSV 和字段映射。
- Goal：生成可复用 Parquet，保留每条记录。
- Allowed actions：改名、转换已验证类型、解析日期、记录解析失败。
- Forbidden actions：样本筛选、缺失填补、单位猜测、缩尾。
- Required checks：行数相同、六位证券代码不丢前导零、原始文件哈希不变。
- Expected outputs：interim/balance.parquet、字段类型与缺失率表。
- Human review conditions：非数字金额、无效日期、单位冲突或原始文件改变。

##### 3.3 合并年末财务报表

本任务解决多表覆盖不一致及口径混入。

- Input：年度视图 balance、income、cashflow。
- Goal：A/B 分列合并，保留两侧未匹配记录。
- Allowed actions：按已确认年末规则取视图、验证键、outer merge、添加来源标记。
- Forbidden actions：inner merge 丢行、按公司名匹配、将 B 填进缺失 A。
- Required checks：每侧键唯一；N_after = matched + master_only + using_only；年度和群体未匹配分布。
- Expected outputs：候选底表、merge.csv、merge_by_year.csv。
- Human review conditions：键不唯一、样本异常膨胀、异常群体集中。

##### 3.4 构造 Leverage 并比较盈利能力口径

本任务解决比率可以计算但定义不透明的问题。

- Input：保留分子分母及 A/B 区分的年度候选表。
- Goal：Leverage；ROA、ROE 期末/平均口径和归母口径候选。
- Allowed actions：按公式计算，零分母返回缺失，保留负权益，精确匹配 t−1。
- Forbidden actions：删除 Leverage > 1、替换极端 ROE、宣布一个口径为唯一正确。
- Required checks：币种/单位、相邻年度、零与近零分母、缺失率、共同样本比较、反算。
- Expected outputs：分子分母、distribution.csv、ratio_common_sample.csv、ratio_correlations.csv。
- Human review conditions：跨表单位不明、公司身份冲突、拟采用某一口径进入正式研究。

##### 3.5 生成 Data Audit

本任务解决“怎样核实 Agent 没有把错误藏在处理过程里”。

- Input：冻结的原始指纹、代码、候选表与各步日志。
- Goal：形成可交给另一位研究者检查的证据包。
- Allowed actions：只读诊断、汇总、在受限目录保存异常行。
- Forbidden actions：边审计边自动修复、向公共 Notebook 输出真实公司记录。
- Required checks：结构、样本流、合并、缺失、分布、会计、时间、身份映射及公开范围。
- Expected outputs：AUDIT_REPORT.md、机器可读汇总、HUMAN REVIEW REQUIRED 清单。
- Human review conditions：任一关键数据定义未确认、审计失败、输入版本发生变化。

#### 4. Audit Rules 与冻结条件

每次交付必须列输入、原始/输出行数、两个层级的主键检查、删除或未纳入理由、缺失率、合并分类、关键分布、会计与时间检查及人工待办。

执行验证通过不自动触发最终冻结：当前允许冻结“本次候选结果与审计证据”，不允许标记“完整 A 股公司年度研究库已验收”。人工裁定后应另起运行版本，不覆盖本轮证据。


## 主案例：建立中国上市公司年度基础数据库 {#sec-financial-data-csmar}

这一节把前面的判断落实到真实文件。目标研究期为 2015–2025 年，财务数据向前保留 2014 年，供平均资产或平均权益口径计算期初值。基础库阶段保留金融行业、ST/PT 和退市公司，不提前实施某篇论文的样本删除。

### 原始文件里实际有什么

先核对文件，再谈能构造什么。本批数据有 10 个 ZIP，覆盖年度基本信息、静态公司信息、三张财务报表、股东、治理、股权性质、年回报率和披露日期；另有基本信息数据库说明书。股东 ZIP 含两个 CSV，属于同一张表的分片，须追加后一起检查。

| 数据表 | 本批原始行数 | 关键发现 |
|---|---:|---|
| 年度基本信息 | 50,706 | 有公司 ID、证券 ID、年度行业；状态定义仍需谨慎 |
| 静态公司基本情况 | 3,194 | 含上市和退市日期，但证券覆盖少于其他表 |
| 资产负债表 | 451,054 | 同时含 A/B、季度末、年末和 1 月 1 日记录 |
| 利润表 | 451,037 | 研发费用字段自 2018 年起使用 |
| 现金流量表 | 451,022 | 与其他财务表不能假定覆盖完全相同 |
| 十大股东，两片合计 | 1,870,326 | 主键需要股东排名；不能直接当公司年度表 |
| 股权性质 | 49,848 | 含 Top1 对照字段及复合性质编码 |
| 治理综合信息 | 49,848 | 本次导出不含现成 Top1 |
| 年个股回报率 | 49,555 | 市场类型可提供 A 股身份证据；市值单位为千 |
| 披露日期 | 197,324 | 文件名虽写预披露，字段实际包含 Actudt |

数字来自本次读取，见[来源清单](../../data/financial-data/SOURCE_MANIFEST.md)和[机器可读结构审计](../../data/financial-data/audit/schema.json)。文件内日期跨度并不完全相同，财务表和披露表还含 2026 年季报；年度视图另按 2014–2025 年末规则生成。

### 标准化之后再取年度视图

这一小节解释哪些操作只是技术整理，哪些操作已经改变数据范围。标准化阶段保留全部行，只进行字段映射和类型处理。年度视图才保留 12 月 31 日，A/B 分开并入。A 表示合并报表，B 表示母公司报表；归母净利润则是合并报表内部的利润归属概念，与母公司单体报表不同。

每一步留下样本流。季度记录没有被当成错误删除；它们仍保存在 interim，只是不属于这一年度视图。1 月 1 日记录也不被直接冒充当年年末。若研究季度动态或期初调整，要另行建立规则。

主表采用各年度来源键的并集，记录每张表是否提供对应观测。这能显露覆盖差异。比如缺少现金流记录的公司不因为一次 inner merge 就消失。不同年份、上市状态和行业中的未匹配情况，需要进一步解释。

### 候选底表与真正的公司年度主表

这一小节处理本次最关键的身份问题。研究期年度并集有 49,078 行；候选上市期间视图有 47,019 行，对应 5,708 个证券代码和 5,702 个公司 ID。它使用市场表确认曾观测到的 A 股身份，并按可用上市日期排除已知上市前记录；未纳入的 2,059 行仍保存在候选层。

这些行数不能直接称为完整 A 股公司总体。静态公司信息覆盖不足，上市日期在两个来源间存在差异；当前未发现已知退市后记录，不等于所有未知退市均已排除。

更直接的检查是：证券代码—年份键没有重复，公司 ID—年份却出现 26 个重复组，共 52 行。部分组财务数值也不同。可能需要证券代码变更或转板历史来解释，不能按代码大小、记录顺序或收益率是否缺失选一条。这里保留全部证据，等待人工核对。

::: {.callout-warning title="本轮底表的状态"}
本轮已完成标准化、候选年度表与真实审计。**最终公司年度 master 尚未验收。** 身份映射、上市日期、完整退市覆盖和历史状态等问题见 [DATA_GAPS.md](../../data/financial-data/DATA_GAPS.md)。程序文件名中的 `master` 不代表这些问题已解决。
:::



## 00. Setup：把数据和代码的位置分开 {#sec-csmar-00 .unnumbered}

将环境变量 `FINECO_CSMAR_ROOT` 设置为含 `raw-zip/` 的授权目录，随后重启内核。请阅读 `data/financial-data/README.md`。本实验不会访问实时 API，不需要账号或密钥。

下面从当前工作目录向上寻找课程根目录，保持路径可迁移。依赖已有环境中的 pandas、numpy、pyarrow、nbformat、nbclient、ipykernel。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
请先读取本项目 AGENTS.md、PROJECT_PLAN.md、DATA_PROTOCOL.md
和 DECISIONS.md。目标是从获授权的 CSMAR ZIP 构造年度候选库。
研究期 2015–2025，2014 年用于相邻期分母支持；目标粒度为公司年度。
先检查输入路径、数据权限、现有成果和可修改范围，列出阶段计划。
不要取数、安装系统软件、读取凭据、覆盖原始文件或推送 Git。
把未确认规则列出来；不要把我的目标理解成允许自行删样本和填补。
交付：输入可用性、文件所有权、执行顺序、每阶段验收与阻塞项。
```

In [1]:
from pathlib import Path
import sys, json, hashlib
from datetime import datetime
import pandas as pd
from IPython.display import display

# 可从仓库根目录或 Notebook 所在目录启动。
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/financial-data/pipeline.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在 FinEco 仓库内部运行 Notebook。")
sys.path.insert(0, str(PROJECT / "tools/financial-data"))
from pipeline import (data_root, source_inventory, standardize, annual_views,
                      build_master, construct_variables, audit_master,
                      require_key, dump_json)
ROOT = data_root()
RUN = ROOT / "financial-data-v2/runs" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
for folder in ["interim", "processed", "audit"]:
    (RUN / folder).mkdir(parents=True, exist_ok=False)
print("输入目录已验证；每轮执行使用独立运行目录。")
print("pandas:", pd.__version__)

输入目录已验证；每轮执行使用独立运行目录。
pandas: 2.3.3


## 01. Source inventory：先确认拿到的是哪一批数据 {#sec-csmar-01 .unnumbered}

文件名相同不保证内容相同。SHA256 是文件内容的指纹；manifest 记录来源 ZIP、成员和大小。学生复跑时应比较指纹，不把不同批次输出直接对比。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
读取已确认输入根的 raw-zip，仅在指定派生目录解压。
记录每个 ZIP 的原名、大小、SHA256、成员、编码和字段说明文件。
逐表阅读说明，报告行列数、实际字段、时间范围、覆盖与候选键。
多份 CSV 分片必须全部列入；追加后重新验证完整键。
不根据字段代码猜含义，不按文件名推断实际公告字段一定不存在。
禁止筛样本和输出受限记录到公开 Notebook。
交付：Source Manifest、schema、字段定义缺口和读取失败明细。
字段无说明、压缩包损坏或键含义不明时标记 HUMAN REVIEW REQUIRED。
```

In [2]:
manifest = source_inventory(ROOT)
dump_json(manifest, RUN / "audit/source_manifest.json")
display(pd.DataFrame(manifest)[["file", "size_bytes", "sha256"]])

,file,size_bytes,sha256
0,上市公司基本信息年度表213006778(仅供中山大学使用).zip,660006,094173795a366b0548a426f3e78f980cc52eaf62f479e4...
1,中国上市公司股权性质文件062310522(仅供中山大学使用).zip,1128717,6112976d4bceb5ded729d75b521c4e687aceb3d164360d...
2,公司基本情况文件060008376退市时间(仅供中山大学使用).zip,288082,b2809505382d1b7a197c76420875a5d9e29df709edcb04...
3,利润表054017684(仅供中山大学使用).zip,24323883,746c5b63ba4728862aa478f7e54b468048c2fcde1ce48a...
4,十大股东文件061816460(仅供中山大学使用).zip,21048413,490e59e9d04dfe24535108256398c3e2b593f3853a60cc...
5,年、中、季报预披露日期表055639123(仅供中山大学使用).zip,1266782,8ce167be0c2f1578ee5a1358e7864881c7ccf79717c5df...
6,年个股回报率文件062802527(仅供中山大学使用).zip,1286420,33c9f111dce1426116ecfac7a50d7ba5886ce62c07cc19...
7,治理综合信息文件060913632(仅供中山大学使用).zip,622431,5f4c1f8b44fc082ea84972a7de957c72061007d21a5b3a...
8,现金流量表(直接法)055248189(仅供中山大学使用).zip,14337912,55cdb111b2643ef24bf0f8cc8954d6d5a4bacff781f833...
9,资产负债表052534564(仅供中山大学使用).zip,38880080,c85797b0ec8ef2706caf1cc6e581e7137bf4740857fc2d...


## 02. Schema audit：先读字段说明，再认识一行记录 {#sec-csmar-02 .unnumbered}

`standardize` 读取 ZIP 内 TXT，并对每个 CSV 字段确认说明存在。股东数据的两个分片都必须读取。日期转换失败会计数；原文保留在受限解压目录。标准化层不筛样本。

这里同时执行标准化，以避免为课堂展示重复读入约数百万条记录；下一节单独检查转换规则。

In [3]:
tables, schemas, fields = standardize(ROOT, RUN)
dump_json(schemas, RUN / "audit/schema.json")
dump_json(fields, RUN / "audit/field_dictionary.json")
schema_table = pd.DataFrame(schemas)
display(schema_table[["table", "rows", "columns", "company_coverage",
                      "candidate_keys", "duplicate_key_rows"]])

,table,rows,columns,company_coverage,candidate_keys,duplicate_key_rows
0,annual_info,50706,14,5741,"[Symbol, EndDate]",0
1,ownership,49848,12,5741,"[Symbol, EndDate]",0
2,company,3194,6,3194,[Stkcd],0
3,income,451037,15,5802,"[Stkcd, Accper, Typrep]",0
4,shareholders,1870326,7,5741,"[Stkcd, Reptdt, S0501b]",0
5,disclosure,197324,5,5803,"[Stkcd, Accper]",0
6,market,49555,7,5711,"[Stkcd, Trdynt]",0
7,governance,49848,7,5741,"[Stkcd, Reptdt]",0
8,cashflow,451022,9,5802,"[Stkcd, Accper, Typrep]",0
9,balance,451054,22,5802,"[Stkcd, Accper, Typrep]",0


读表时比较 `candidate_keys`，而不只看“重复数是 0”。财务表需要证券代码、报表日和报表类型；股东表还需要排名。较细粒度的键唯一，不意味着公司年度键唯一。

## 03. Standardization：字段名变了，经济含义不能变 {#sec-csmar-03 .unnumbered}

证券代码保留字符串，避免丢失前导零；财务金额保留原始数值单位。市值字段说明明确使用“千”，年度层才转换为元。股东比例原值是百分数，`top1` 才转为 0–1 比例。

 本批财务 TXT 没有明确标出各金额的币种与单位，跨表 ROA/ROE 暂为条件性候选；应补核财务数据库说明或导出设置。原始空白值不会填成零。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
按已核验映射统一字段名和类型，代码保留字符串，解析日期。
单位有明确依据才换算，保留转换规则与原始值来源。
interim 保留全部行，报告类型/日期转换失败及前后行数。
再单独建立 2014–2025 年 12 月 31 日年度视图，A/B 口径分别保留。
不是年末的记录仍留在 interim。不要自动填缺失、缩尾或删公司。
交付：Parquet、字段映射、键检查、样本流与转换失败汇总。
若年度键重复，先解释报表、版本与粒度，不运行任意去重。
```

In [4]:
# 展示字段类型和缺失率，不展示任何公司记录。
balance = tables["balance"]
display(pd.DataFrame({"dtype": balance.dtypes.astype(str),
                      "missing_rate": balance.isna().mean()}).loc[
    ["stock_code", "report_date", "Typrep", "total_assets", "total_equity"]])
assert len(balance) == next(s["rows"] for s in schemas if s["table"] == "balance")

,dtype,missing_rate
stock_code,string,0.000000
report_date,datetime64[ns],0.000000
Typrep,string,0.000000
total_assets,float64,0.000011
total_equity,float64,0.000004


## 04. Merge planning：年度筛选和删除重复是不同操作 {#sec-csmar-04 .unnumbered}

财务数据包含季度末、年末、1 月 1 日和 A/B 报表。年度视图保留 2014–2025 年 12 月 31 日，A/B 继续分别保存。其他行仍在 interim，未被销毁。

下面的样本流描述“原表到年度视图”的行数变化，并非论文研究样本排除。

In [5]:
views, flow = annual_views(tables)
display(pd.DataFrame(flow))
# 验证同一日下合并报表与母公司报表是两种口径。
display(views["balance"].groupby("Typrep").agg(
    rows=("stock_code", "size"), securities=("stock_code", "nunique")))

,step,N_before,Removed,N_after,Reason
0,annual_info:年度视图,50706,0,50706,保留 2014–2025 年末；其余行仍保存在 interim
1,ownership:年度视图,49848,0,49848,保留 2014–2025 年末；其余行仍保存在 interim
2,income:年度视图,451037,349071,101966,保留 2014–2025 年末；其余行仍保存在 interim
3,shareholders:年度视图,1870326,1369161,501165,保留 2014–2025 年末；其余行仍保存在 interim
4,disclosure:年度视图,197324,147367,49957,保留 2014–2025 年末；其余行仍保存在 interim
5,market:年度视图,49555,0,49555,保留 2014–2025 年末；其余行仍保存在 interim
6,governance:年度视图,49848,0,49848,保留 2014–2025 年末；其余行仍保存在 interim
7,cashflow:年度视图,451022,349058,101964,保留 2014–2025 年末；其余行仍保存在 interim
8,balance:年度视图,451054,349086,101968,保留 2014–2025 年末；其余行仍保存在 interim


,rows,securities
Typrep,,
A,51484,5741
B,50484,5691


In [6]:
# 与数据管理讲义对应：只连接年度 A 口径，B 仍保留在 views。
left = views["balance"].query("Typrep == 'A'")
right = views["income"].query("Typrep == 'A'")
keys = ["stock_code", "fiscal_year"]
require_key(left, keys, "资产负债表 A")
require_key(right, keys, "利润表 A")
merged = left.merge(right, on=keys, how="outer", validate="one_to_one",
                    indicator=True, suffixes=("_balance", "_income"))
display(merged["_merge"].value_counts().rename("N").to_frame())
assert not merged.duplicated(keys).any()

,N
_merge,
both,51484
left_only,0
right_only,0


### 小型合并实验：同样能运行，行数却不同

以下为明确虚构的教学数据。一家公司一年的资产不能通过只按公司连接而变成两条独立年度观测。先把日度量聚合为研究所需的年度量，才有共同粒度。这里示范年均换手率，不把日收益的算术均值误称年度持有收益率。

In [7]:
# 改编自《数据分析与经济决策》课程 data_manage，全部为教学数值。
annual_demo = pd.DataFrame({"firm": ["A", "B"], "year": [2020, 2020], "assets": [100, 200]})
daily_demo = pd.DataFrame({"firm": ["A", "A", "B"], "year": [2020]*3,
                           "turnover": [1., 3., 2.]})
wrong = annual_demo.merge(daily_demo, on=["firm", "year"])
aggregated = daily_demo.groupby(["firm", "year"], as_index=False).turnover.mean()
correct = annual_demo.merge(aggregated, on=["firm", "year"], validate="one_to_one")
print({"原年度行数": len(annual_demo), "直接合并行数": len(wrong), "聚合后行数": len(correct)})
display(correct)
assert len(correct) == 2 and len(wrong) == 3

{'原年度行数': 2, '直接合并行数': 3, '聚合后行数': 2}


,firm,year,assets,turnover
0,A,2020,100,2.0
1,B,2020,200,2.0


## 05. Firm-year master：先保留覆盖差异，再核查公司身份 {#sec-csmar-05 .unnumbered}

流水线使用年度表键的并集，防止选某一张财务表作为母表时静默丢失其他表的公司。A/B 财务列分开，所有合并使用 `validate="one_to_one"`。

合并成功仅证明证券代码—年度键没有膨胀。公司 ID 可能对应不同证券代码，还必须另外审计。存在冲突时不自行选代码、相加或取均值。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
先说明两侧表粒度、键和目标底表粒度，检查空键与重复键。
采用能保留覆盖差异的连接，并为每次合并设置预期连接关系。
A/B 财务列分开，不用一类填补另一类；记录 matched、仅左、仅右，
以及 N_before、N_after 和分年匹配表。未匹配不自动删除。
分别检查证券年度键和公司年度键。遇到一公司多证券，保留全部
候选并生成本地冲突表；不能按代码大小、收益是否缺失来选一条。
交付候选底表、merge audit、身份冲突与总体覆盖限制。
未解决公司身份时，不能宣称已得到唯一公司年度 master。
```

In [8]:
base, merge_audit, merge_by_year = build_master(views)
display(pd.DataFrame(merge_audit))
require_key(base, ["stock_code", "fiscal_year"], "年度候选底表")
print("公司年度冲突行数 (含 2014 支持层):", int(base.firm_year_conflict.sum()))

,table,N_before,N_after,matched,master_only,using_only,duplicates
0,balance_A,50706,51485,50705,1,779,0
1,balance_B,51485,51485,50484,1001,0,0
2,income_A,51485,51485,51484,1,0,0
3,income_B,51485,51485,50482,1003,0,0
4,cashflow_A,51485,51485,51483,2,0,0
5,cashflow_B,51485,51485,50481,1004,0,0
6,market,51485,51778,49262,2223,293,0
7,ownership,51778,51778,49848,1930,0,0
8,governance,51778,51778,49848,1930,0,0
9,disclosure,51778,51778,49957,1821,0,0


公司年度冲突行数 (含 2014 支持层): 90


## 06. Variable construction：每个比率都保留分子和分母 {#sec-csmar-06 .unnumbered}

Leverage 使用合并报表的负债/资产。ROA 同时计算净利润/期末资产、净利润/平均资产，以及归母净利润的候选口径；ROE 区分总净利润/总权益和归母净利润/归母权益。负权益不删，零分母返回缺失。

平均资产必须使用相邻年度。缺少 t−1 时不把更早一年当作上一年。2014 数据供 2015 分母使用。以下计算结果仍依赖跨表币种与单位确认。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
按 DATA_PROTOCOL 构造 Leverage 及期末/平均分母 ROA、ROE 候选。
保留所有原始分子分母、币种/单位状态、报表类型与相邻年度匹配。
2015 年平均分母只可取 2014 年，不得用更早可用年份补齐。
零分母返回缺失并计数，负分母与极端比率保留供诊断。
由股东排名 1 的源持股比例构造 Top1，与可比现成字段核验；
没有总股本分母时不得声称从持股数独立重算。
同时报告全样本覆盖和共同样本比较，不替教师选择最终定义。
单位不明确时标明条件性候选，禁止将相关系数当成口径正确证据。
```

In [9]:
base = construct_variables(base)
ratio_cols = [c for c in base if c.startswith(("roa_", "roe_", "leverage_"))]
print("候选比率：", ratio_cols)
assert not base.duplicated(["stock_code", "fiscal_year"]).any()

候选比率： ['leverage_consolidated', 'roa_eop_consolidated', 'roa_parent_income_eop_candidate', 'roa_avg_consolidated', 'roa_parent_income_avg_candidate', 'roe_eop_consolidated', 'roe_avg_consolidated', 'roe_parent_eop_consolidated', 'roe_parent_avg_consolidated']


## 07. Missing / duplicate / outlier：诊断之后才决定 {#sec-csmar-07 .unnumbered}

研发费用字段从 2018 年起使用，早期缺失不能自动解读为没有研发。极端 ROE 可能来自接近零的权益，也可能来自分子异常；诊断阈值不是删除规则。源上市状态字段的定义不能支持完整历史 ST/PT 路径，因此标识保留 `_source` 后缀。

下面只显示汇总，不输出公司名称、证券代码或真实单行数据。

In [10]:
research = base[base.fiscal_year.between(2015, 2025) & base.eligible_known]
display(research.groupby("fiscal_year").rd_expense.agg(
    N="size", nonmissing="count"))
print("权益绝对值不足资产 1% 的行数:",
      int((research.total_equity.abs() / research.total_assets.abs()).lt(.01).sum()))
print("Leverage 大于 1 的行数:", int(research.leverage_consolidated.gt(1).sum()))

,N,nonmissing
fiscal_year,,
2015,2816,0
2016,3040,4
2017,3475,14
2018,3580,3052
2019,3772,3282
2020,4202,3718
2021,4709,4246
2022,5110,4662
2023,5381,4955


权益绝对值不足资产 1% 的行数: 48
Leverage 大于 1 的行数: 290


## 08. Data Audit：让结果接受反向检查 {#sec-csmar-08 .unnumbered}

会计检查使用 max(1 个原始单位, 资产绝对值×10⁻⁸) 作为诊断容差；不符合时保留记录供追查。日期审计检查实际披露是否早于报表日，但不能证明下载的是当时可得版本；财报更正历史仍需单独获取。

报告中的 `master_rows` 是程序候选视图行数，程序文件名里的 master 不代表已经通过公司年度身份审核。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
审计当前候选库的结构、样本流、合并、缺失、分布、分母、
会计关系和信息日期。关键变量报告 N、mean、sd、分位数与极值。
按年和可识别群体比较未匹配，用组内样本量作分母。
区分预约日和实际公告日，检查财报修订版本是否可识别。
会计容差必须写清，超出容差只标识，不自动删行。
仅公开不构成再分发的汇总；具体公司问题留在授权本地。
交付审计报告、证据路径、影响及 HUMAN REVIEW REQUIRED 列表。
每项结论区分已验证、条件性成立、缺资料和待教师决定。
```

In [11]:
summary = audit_master(base, schemas, merge_audit, merge_by_year, flow, RUN)
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
display(pd.read_csv(RUN / "audit/ratio_common_sample.csv"))
display(pd.read_csv(RUN / "audit/distribution.csv").round(5))

,value
candidate_rows,49078
master_rows,47019
securities,5708
firm_ids,5702
missing_firm_id,0
firm_year_duplicate_rows,52
unresolved_candidate_rows,2059
pre_listing_rows,2059
post_delisting_rows,0
a_share_identity_unconfirmed_rows,32


,family,N_common,mean_eop,mean_avg,median_eop,median_avg,correlation
0,roa,44383,0.017447,0.025873,0.029702,0.031178,0.644684
1,roe,44383,0.023625,0.026189,0.057896,0.060038,0.003548


,variable,N,mean,sd,min,p1,p25,p50,p75,p99,max
0,leverage_consolidated,46728.0,4.343100e-01,8.830700e-01,8.360000e-03,5.569000e-02,2.496800e-01,4.087700e-01,5.778900e-01,9.617800e-01,1.783455e+02
1,roa_eop_consolidated,46728.0,1.998000e-02,2.827100e-01,-3.068823e+01,-3.549100e-01,7.840000e-03,3.177000e-02,6.243000e-02,1.918900e-01,7.446080e+00
2,roa_parent_income_eop_candidate,46728.0,1.939000e-02,2.736400e-01,-3.068823e+01,-3.459700e-01,7.370000e-03,2.976000e-02,6.008000e-02,1.861600e-01,7.446080e+00
3,roa_avg_consolidated,44383.0,2.587000e-02,1.227300e-01,-9.116920e+00,-3.035000e-01,7.110000e-03,3.118000e-02,6.397000e-02,2.125200e-01,1.221107e+01
4,roa_parent_income_avg_candidate,44383.0,2.480000e-02,1.205200e-01,-9.116920e+00,-2.962800e-01,6.710000e-03,2.916000e-02,6.124000e-02,2.061100e-01,1.207795e+01
5,roe_eop_consolidated,46728.0,2.716000e-02,5.023190e+00,-2.073971e+02,-1.386420e+00,1.757000e-02,6.051000e-02,1.050500e-01,4.326900e-01,9.436241e+02
6,roe_avg_consolidated,44383.0,2.619000e-02,1.677260e+00,-1.748947e+02,-1.096130e+00,1.538000e-02,6.004000e-02,1.100000e-01,4.421900e-01,8.432795e+01
7,roe_parent_eop_consolidated,46728.0,-5.110000e-03,2.086360e+00,-2.350960e+02,-1.556730e+00,1.767000e-02,6.052000e-02,1.052700e-01,4.722100e-01,1.211421e+02
8,roe_parent_avg_consolidated,44383.0,-1.356000e-02,6.617090e+00,-1.305214e+03,-1.176290e+00,1.552000e-02,5.990000e-02,1.102300e-01,4.768200e-01,1.259991e+02
9,annual_stock_return,43845.0,1.074400e-01,5.594700e-01,-9.843600e-01,-6.176900e-01,-2.130400e-01,-5.550000e-03,2.648800e-01,2.175800e+00,1.820616e+01


看 `ratio_common_sample` 的 `N_common`：只有使用同一批可用观测，均值差异才主要反映分母定义的变化。全样本描述统计同时混入缺失造成的样本变化。相关系数高也不代表两种定义可互换。

## 09. Agent workflow：先写任务边界，再运行验证 {#sec-csmar-09 .unnumbered}

可复制任务：读取标准化资产负债表；按证券代码、日期和报表类型检查键；允许统计和列出本地待核对记录；禁止删除、填补、缩尾或选择报表类型；输出键检查、缺失率及重复组数；发现非唯一键时标记 HUMAN REVIEW REQUIRED。

完整规则见 [DATA_PROTOCOL.md](../../data/financial-data/DATA_PROTOCOL.md)，分阶段提示词见 [PROMPTS.md](../../data/financial-data/agent-project/PROMPTS.md)。下面的 verification cells 是学生核验 Agent 的最小起点，不能代替全部审计。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
请根据指定审计问题定位最早出错步骤，先报告原因与受影响输出。
区分实现错误、源数据限制与研究选择，不为让检查通过而改阈值。
只修复有明确依据的实现错误，保留修订前后输入哈希与样本比较。
修复后从头执行相关流水线和 Notebook，比较键、行数、缺失、
匹配及变量分布，说明哪些变化是预期的。
仍缺身份、单位或历史状态时，不自行补造，继续保留待裁定。
交付修改说明、重新执行证据、差异表和剩余问题。
```

In [12]:
require_key(views["balance"], ["stock_code", "fiscal_year", "Typrep"], "年度财务")
assert all(x["N_after"] == x["matched"] + x["master_only"] + x["using_only"]
           for x in merge_audit)
assert summary["duplicate_stock_year"] == 0
# 不把已知待审事项伪装为通过的断言。
print("HUMAN REVIEW REQUIRED:",
      {"公司年度重复行": summary["firm_year_duplicate_rows"],
       "上市日期冲突行": summary["listing_date_conflicts"],
       "实际披露缺失行": summary["actual_announcement_missing"]})

HUMAN REVIEW REQUIRED: {'公司年度重复行': 52, '上市日期冲突行': 409, '实际披露缺失行': 314}


## 10. Export：冻结输入、代码与执行证据 {#sec-csmar-10 .unnumbered}

完整 interim、候选底表、待审记录已经写入授权数据根目录下的独立运行目录；它们不随 GitHub 或网页分发。保存脚本哈希和版本后，才能知道本次结果对应哪段代码。

本 Notebook 不自动发布汇总文件；维护脚本只按固定允许清单复制已检查的汇总表到公开目录。

### 本阶段给 Codex 的任务

以下为教学任务模板。先核对路径和本项目规则，再复制使用；后续代码和输出展示本案例的实现。

```text
请保存本次输入指纹、脚本哈希、环境版本、真实 Notebook 输出、
规则与决策记录。检查公开目录和 Notebook 不含受限公司记录、
账号、密钥或下载密码。完整数据仅留在授权路径。
列出可复跑命令、文件依赖、审计通过项和未决事项。
分别报告执行验证、最终公司年度数据、网页预览和公开发布状态。
未决问题影响最终口径时，只保存候选版本与证据，不标正式冻结。
不提交、推送或覆盖已有发布结果，除非另有明确授权。
```

In [13]:
import numpy as np
dump_json({"executed_at": datetime.now().isoformat(), "pandas": pd.__version__, "numpy": np.__version__,
           "pipeline_sha256": hashlib.sha256((PROJECT / "tools/financial-data/pipeline.py").read_bytes()).hexdigest()},
          RUN / "audit/execution.json")
(ROOT / "financial-data-v2/latest-run.txt").write_text(str(RUN), encoding="utf-8")
print("候选版本与执行证据已保存；最终公司年度数据尚待人工裁定。")

候选版本与执行证据已保存；最终公司年度数据尚待人工裁定。


## 11. Exercises：把运行成功转成判断依据 {#sec-csmar-11 .unnumbered}

1. 找到一种原表键唯一、公司年度键却不唯一的情况，解释为什么不能 `drop_duplicates`。
2. 使用 `merge_by_year.csv` 和 `unmatched_groups.csv`，比较新上市与其他公司的未匹配率。分母必须使用组内样本数。
3. 解释 ROA 平均资产口径为什么少于期末口径；在共同样本上比较两者。
4. 如果想删除 ST，当前 `_source` 标识是否足够？写出缺少的资料与审计规则。
5. 将真实极端 ROE 的诊断改写成一个全新教学数值例子，明确标记教学示例，不公开原始行。

练习完成后交付修改过的 Task Rules、汇总审计和解释；不上传 CSMAR 数据。

In [14]:
# 练习起点：先比较未匹配率，再讨论原因；本单元格只读汇总表。
groups = pd.read_csv(RUN / "audit/unmatched_groups.csv")
groups["unmatched_rate"] = groups["unmatched"] / groups["N"]
display(groups[groups["group"].eq("newly_listed")].round(4))

,group,value,source,N,unmatched,unmatched_rate
0,newly_listed,False,market,43960,53,0.0012
1,newly_listed,False,disclosure,43960,255,0.0058
2,newly_listed,False,top1,43960,311,0.0071
3,newly_listed,False,balance_A,43960,291,0.0066
4,newly_listed,False,income_A,43960,291,0.0066
5,newly_listed,False,cashflow_A,43960,291,0.0066
6,newly_listed,True,market,3059,3,0.0010
7,newly_listed,True,disclosure,3059,3,0.0010
8,newly_listed,True,top1,3059,43,0.0141
9,newly_listed,True,balance_A,3059,0,0.0000


## 教学小表：不依赖 CSMAR 的独立实验 {#sec-csmar-mini}

本节代码可以独立运行，数值全部为虚构教学例子。依次验证读取类型、错误/正确合并、追加、宽长往返、Parquet 类型保存、SQL/pandas 对照，以及缺失填零与比率手算。它对应前三章的小例子，不代表真实公司。

读取例子重点看证券代码的前导零；合并表比较 2→3 与 2→2；SQL 结果与 pandas 逐项相等；研发率两种均值显示填零改变了假设。每项均有可检查的断言。


In [15]:
"""三章理论的小表例子。全部数值为虚构教学数据，不需要 CSMAR。"""
from io import StringIO, BytesIO
import sqlite3
import numpy as np
import pandas as pd
from IPython.display import display


def run_examples():
    # 明确类型读取，保留前导零；先看结构，再做计算。
    source = StringIO('code,date,assets\n000001,2020-12-31,100\n000002,2020-12-31,200\n')
    read_demo = pd.read_csv(source, dtype={'code': 'string'}, parse_dates=['date'])
    assert read_demo.code.iloc[0] == '000001'
    display(read_demo)
    display(read_demo.dtypes.astype(str).rename('dtype').to_frame())

    # 与理论篇对应的两公司、三交易日小表，绝非真实公司记录。
    annual = pd.DataFrame({'firm': ['A', 'B'], 'year': [2020, 2020], 'assets': [100., 200.]})
    company = pd.DataFrame({'firm': ['A', 'B'], 'industry': ['甲行业', '乙行业']})
    daily = pd.DataFrame({'firm': ['A', 'A', 'B'],
                          'date': pd.to_datetime(['2020-01-02', '2020-01-03', '2020-01-02']),
                          'turnover': [1., 3., 2.]})
    daily['year'] = daily.date.dt.year
    wrong = annual.merge(daily, on=['firm', 'year'])
    yearly = daily.groupby(['firm', 'year'], as_index=False).turnover.mean()
    correct = annual.merge(yearly, on=['firm', 'year'], validate='one_to_one', indicator=True)
    assert len(wrong) == 3 and len(correct) == 2
    assert correct.turnover.tolist() == [2., 2.]
    display(pd.DataFrame({'操作': ['年度输入', '直接连接', '聚合后连接'], '行数': [2, len(wrong), len(correct)]}))
    display(correct)

    # append 保留来源；同一片段重复追加时应被键检查发现。
    appended = pd.concat([annual.iloc[:1].assign(source='part1'),
                          annual.iloc[1:].assign(source='part2')], ignore_index=True)
    assert not appended.duplicated(['firm', 'year']).any()
    duplicated = pd.concat([appended, appended.iloc[:1]], ignore_index=True)
    assert duplicated.duplicated(['firm', 'year'], keep=False).sum() == 2

    # 宽长转换只重排信息，不能自动聚合重复公司年度。
    wide = pd.DataFrame({'firm': ['A', 'B'], 'assets_2019': [90., 180.], 'assets_2020': [100., 200.]})
    long = wide.melt(id_vars='firm', var_name='period', value_name='assets')
    long['year'] = long.period.str[-4:].astype(int)
    restored = long.pivot(index='firm', columns='period', values='assets').reset_index()
    pd.testing.assert_frame_equal(wide, restored, check_names=False)
    display(long[['firm', 'year', 'assets']].sort_values(['firm', 'year']))

    # Parquet 保存与读取类型；使用内存字节流，不改写任何原始文件。
    buffer = BytesIO()
    read_demo.to_parquet(buffer, index=False)
    buffer.seek(0)
    restored_types = pd.read_parquet(buffer)
    pd.testing.assert_frame_equal(read_demo, restored_types)

    # SQLite 唯一索引是显式设置的，不是 to_sql 自动识别业务键。
    with sqlite3.connect(':memory:') as conn:
        annual.to_sql('annual', conn, index=False)
        company.to_sql('company', conn, index=False)
        conn.execute('CREATE UNIQUE INDEX annual_key ON annual(firm, year)')
        conn.execute('CREATE UNIQUE INDEX company_key ON company(firm)')
        query = '''SELECT c.industry, COUNT(*) AS n, AVG(f.assets) AS mean_assets
                   FROM annual AS f JOIN company AS c ON f.firm = c.firm
                   WHERE f.year = 2020 GROUP BY c.industry ORDER BY c.industry'''
        sql_result = pd.read_sql_query(query, conn)
    pandas_result = (annual.merge(company, on='firm', validate='many_to_one')
                     .groupby('industry', as_index=False)
                     .agg(n=('firm', 'size'), mean_assets=('assets', 'mean')))
    pd.testing.assert_frame_equal(sql_result, pandas_result)
    display(sql_result)

    # 缺失填零改变了假设和统计分母；这里只比较，不作处理推荐。
    rd = pd.Series([0., np.nan, 2.]) / 100
    assert np.isclose(rd.mean(), .01)
    assert np.isclose(rd.fillna(0).mean(), 2/300)
    # 算术和复合回报、股利回报、盈利比率均为手算可核对的小例子。
    comparisons = pd.DataFrame({
        '指标': ['研发率_已知样本均值', '研发率_填零后均值',
                 '两日算术平均收益', '两日复合收益', '含股利回报',
                 '期末资产ROA', '平均资产ROA', '近零权益ROE'],
        '数值': [rd.mean(), rd.fillna(0).mean(), np.mean([.1, -.1]),
                 np.prod(1 + np.array([.1, -.1])) - 1,
                 (9 + 1) / 10 - 1, 10/200, 10/150, 1/.01]})
    display(comparisons.round(6))
    assert np.isclose(comparisons.loc[3, '数值'], -.01)
    print('教学小表检查通过：读取、合并、追加、宽长往返、类型保存、SQL 对照与手算。')



run_examples()


,code,date,assets
0,000001,2020-12-31,100
1,000002,2020-12-31,200


,dtype
code,string
date,datetime64[ns]
assets,int64


,操作,行数
0,年度输入,2
1,直接连接,3
2,聚合后连接,2


,firm,year,assets,turnover,_merge
0,A,2020,100.0,2.0,both
1,B,2020,200.0,2.0,both


,firm,year,assets
0,A,2019,90.0
2,A,2020,100.0
1,B,2019,180.0
3,B,2020,200.0


,industry,n,mean_assets
0,乙行业,1,200.0
1,甲行业,1,100.0


,指标,数值
0,研发率_已知样本均值,0.010000
1,研发率_填零后均值,0.006667
2,两日算术平均收益,0.000000
3,两日复合收益,-0.010000
4,含股利回报,0.000000
5,期末资产ROA,0.050000
6,平均资产ROA,0.066667
7,近零权益ROE,100.000000


教学小表检查通过：读取、合并、追加、宽长往返、类型保存、SQL 对照与手算。


## 如何阅读本轮结果与继续返修 {#sec-csmar-review}

第 08 节的 summary 表是验收入口。证券年度键唯一与公司年度键唯一分别检查：本批研究视图为 47,019 行，公司年度存在 26 组重复、52 行，涉及 6 个公司 ID。不能任意删除一条让唯一性通过。

上市日期冲突为 409 行，实际公告缺失为 314 行，会计差异超出容差为 24 行。它们属于不同问题：日期定义要查资料，会计差异要核原表，身份要补映射历史。源字段的最新上市状态不能自动解释为历年 ST/PT。

ratio_common_sample 表对同一批 44,383 行比较 ROA。期末与平均资产口径均值约 1.74% 与 2.59%，相关系数约 0.645。这是暂定证券年度视图的条件性比较，单位和身份仍未冻结。Top1 的 46,664 个可比记录没有超过 0.01 个百分点的差异；相符不等于独立年报验证。

返修时使用 P06：先追到最早出错步骤，再修改有证据的实现问题；缺少资料的事项进入 DECISIONS，不让 Agent 编造选择。P07 保存候选证据，并分别声明执行、研究数据、网页和发布状态。

学生最终提交规则、使用过的提示词、可执行代码、汇总审计和决策说明，不提交受限公司记录。完整未决项见 [DATA_GAPS](../../data/financial-data/DATA_GAPS.md)，本次汇总见 [AUDIT_REPORT](../../data/financial-data/AUDIT_REPORT.md)。
